# Notebook 124: GRUと時系列予測
## Gated Recurrent Unit & Time Series Forecasting
---
### このノートブックの位置づけ
**Unit 8「シーケンスモデリング」** の実践編。LSTMの簡略版であるGRUを学び、RNN/LSTM/GRUの三者を実践的な時系列予測タスクで比較します。

### 学習目標
1. **GRUの2ゲート構造** （リセットゲート・更新ゲート）を理解し実装できる
2. **RNN/LSTM/GRU** の三者をパラメータ数・学習速度・精度で比較できる
3. **時系列データの前処理** （正規化、ウィンドウ作成）を実装できる
4. **多ステップ予測** を実装し、誤差の累積を観察できる

### 前提知識
- Notebook 121-123（バニラRNN、BPTT、LSTM）

⏱️ **推定学習時間**: 120-150分  
📊 **難易度**: ★★★☆☆（中級）  
🎓 **カテゴリ**: 実践

## 目次

1. [GRUの設計思想](#1-gruの設計思想)
2. [GRU実装](#2-gru実装)
3. [RNN/LSTM/GRU 三者比較](#3-rnnlstmgru-三者比較)
4. [時系列データの前処理](#4-時系列データの前処理)
5. [合成時系列予測](#5-合成時系列予測)
6. [多ステップ予測](#6-多ステップ予測)
7. [まとめ](#7-まとめ)
8. [自己評価クイズ](#8-自己評価クイズ)

In [ ]:
# ============================================================
# セットアップ
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# 日本語フォント設定（利用可能な場合）
try:
    plt.rcParams['font.family'] = 'DejaVu Sans'
except:
    pass

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("NumPy version:", np.__version__)
print("Setup complete!")

---
## 1. GRUの設計思想

### LSTMからGRUへ：なぜ簡略化するのか？

LSTMは**4つのゲート**（入力ゲート、忘却ゲート、出力ゲート、セル状態候補）を持ち、非常に強力ですが：
- パラメータ数が多い → 学習が遅い
- 構造が複雑 → 過学習しやすい（小さなデータセット）
- 推論コストが高い

**GRU（Gated Recurrent Unit, Cho et al., 2014）** は2つのゲートに簡略化：

| ゲート | 記号 | 役割 |
|--------|------|------|
| **更新ゲート** (Update Gate) | $z_t$ | 前の隠れ状態をどれだけ保持するか |
| **リセットゲート** (Reset Gate) | $r_t$ | 前の隠れ状態をどれだけ忘れるか |

### GRUの数式

$$z_t = \sigma(W_z [h_{t-1}, x_t] + b_z) \quad \text{(更新ゲート)}$$

$$r_t = \sigma(W_r [h_{t-1}, x_t] + b_r) \quad \text{(リセットゲート)}$$

$$\tilde{h}_t = \tanh(W_h [r_t \odot h_{t-1}, x_t] + b_h) \quad \text{(候補状態)}$$

$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t \quad \text{(最終状態)}$$

### 直感的理解

- **更新ゲート $z_t$** は LSTMの忘却ゲート＋入力ゲートを1つに統合したもの
  - $z_t \approx 1$：新しい候補状態を採用（「更新する」）
  - $z_t \approx 0$：前の隠れ状態をそのまま保持（「記憶を維持」）
- **リセットゲート $r_t$** は候補状態の計算時に過去の情報をリセット
  - $r_t \approx 0$：過去を忘れて、入力のみから候補を作る
  - $r_t \approx 1$：過去の情報を完全に使う
- **セル状態と隠れ状態が統合** されている → LSTMより構造がシンプル

In [ ]:
# ============================================================
# GRU vs LSTM ゲート構造の比較図
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# --- LSTM構造図 ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.set_title('LSTM: 4 Gates', fontsize=16, fontweight='bold')
ax.axis('off')

# LSTMのゲート
lstm_gates = [
    (2, 7, 'Forget Gate\n(f_t)', '#FF6B6B'),
    (5, 7, 'Input Gate\n(i_t)', '#4ECDC4'),
    (8, 7, 'Output Gate\n(o_t)', '#45B7D1'),
    (5, 4, 'Cell Candidate\n(~c_t)', '#96CEB4'),
]

for x, y, label, color in lstm_gates:
    box = FancyBboxPatch((x-1, y-0.7), 2, 1.4, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='black', linewidth=2, alpha=0.8)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold')

# 状態
for x, y, label in [(5, 1.5, 'Cell State (c_t)'), (5, 9.2, 'Hidden State (h_t)')]:
    ax.text(x, y, label, ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# 矢印
ax.annotate('', xy=(5, 8.7), xytext=(5, 7.7), arrowprops=dict(arrowstyle='->', lw=2))
ax.annotate('', xy=(5, 3.3), xytext=(5, 2.2), arrowprops=dict(arrowstyle='->', lw=2))
ax.text(1, 2, 'Parameters: 4(nh + nx)h + 4h\n= many', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#FFE0E0', alpha=0.8))

# --- GRU構造図 ---
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.set_title('GRU: 2 Gates', fontsize=16, fontweight='bold')
ax.axis('off')

# GRUのゲート
gru_gates = [
    (3, 7, 'Update Gate\n(z_t)', '#FF6B6B'),
    (7, 7, 'Reset Gate\n(r_t)', '#4ECDC4'),
    (5, 4, 'Candidate\n(~h_t)', '#96CEB4'),
]

for x, y, label, color in gru_gates:
    box = FancyBboxPatch((x-1, y-0.7), 2, 1.4, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='black', linewidth=2, alpha=0.8)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold')

# 状態（セル状態なし！）
ax.text(5, 9.2, 'Hidden State (h_t) ONLY', ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
ax.text(5, 1.5, 'No Cell State!', ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='#E8E8E8', alpha=0.8),
        color='gray', style='italic')

# 矢印
ax.annotate('', xy=(5, 8.7), xytext=(5, 7.7), arrowprops=dict(arrowstyle='->', lw=2))
ax.annotate('', xy=(5, 3.3), xytext=(5, 4.7), arrowprops=dict(arrowstyle='->', lw=2, color='green'))
ax.text(1, 2, 'Parameters: 3(nh + nx)h + 3h\n= 75% of LSTM', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#E0FFE0', alpha=0.8))

plt.tight_layout()
plt.suptitle('LSTM vs GRU: Gate Structure Comparison', fontsize=18, fontweight='bold', y=1.02)
plt.show()

print("\n【ポイント】")
print("  LSTM: 忘却ゲート、入力ゲート、出力ゲート、セル候補 → 4つの重み行列")
print("  GRU:  更新ゲート、リセットゲート、候補状態 → 3つの重み行列")
print("  → GRUはLSTMの約75%のパラメータ数で同等の性能を発揮できる場合が多い")

In [ ]:
# ============================================================
# セクション2: GRU実装
# ============================================================

# 共通の活性化関数
def sigmoid(x):
    """数値的に安定なシグモイド関数"""
    x = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_derivative(s):
    """シグモイドの出力sから導関数を計算"""
    return s * (1 - s)

def tanh_derivative(t):
    """tanhの出力tから導関数を計算"""
    return 1 - t ** 2


class GRUCell:
    """GRUセル：フォワードパスとバックワードパスの完全実装"""
    
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim
        scale = np.sqrt(2.0 / concat_dim)
        
        # 更新ゲートの重み
        self.Wz = np.random.randn(concat_dim, hidden_dim) * scale
        self.bz = np.zeros(hidden_dim)
        
        # リセットゲートの重み
        self.Wr = np.random.randn(concat_dim, hidden_dim) * scale
        self.br = np.zeros(hidden_dim)
        
        # 候補状態の重み
        self.Wh = np.random.randn(concat_dim, hidden_dim) * scale
        self.bh = np.zeros(hidden_dim)
        
        # キャッシュ（バックワード用）
        self.cache = None
    
    def forward(self, x_t, h_prev):
        """
        フォワードパス
        x_t: (batch_size, input_dim)
        h_prev: (batch_size, hidden_dim)
        returns: h_next (batch_size, hidden_dim)
        """
        # [h_{t-1}, x_t] を結合
        concat = np.concatenate([h_prev, x_t], axis=1)
        
        # 更新ゲート: z_t = sigma(Wz * [h, x] + bz)
        z = sigmoid(concat @ self.Wz + self.bz)
        
        # リセットゲート: r_t = sigma(Wr * [h, x] + br)
        r = sigmoid(concat @ self.Wr + self.br)
        
        # 候補状態: h_cand = tanh(Wh * [r * h, x] + bh)
        concat_r = np.concatenate([r * h_prev, x_t], axis=1)
        h_cand = np.tanh(concat_r @ self.Wh + self.bh)
        
        # 最終状態: h_t = (1 - z) * h_{t-1} + z * h_cand
        h_next = (1 - z) * h_prev + z * h_cand
        
        # バックワード用にキャッシュ
        self.cache = (x_t, h_prev, z, r, h_cand, concat, concat_r)
        return h_next
    
    def backward(self, dh_next):
        """
        バックワードパス（完全実装）
        dh_next: (batch_size, hidden_dim) - h_nextに対する勾配
        returns: dx_t, dh_prev, grads
        """
        x_t, h_prev, z, r, h_cand, concat, concat_r = self.cache
        batch_size = x_t.shape[0]
        
        # h_t = (1 - z) * h_prev + z * h_cand
        dh_prev_direct = dh_next * (1 - z)
        dh_cand = dh_next * z
        dz = dh_next * (h_cand - h_prev)
        
        # h_cand = tanh(...) → tanh微分
        dh_cand_raw = dh_cand * tanh_derivative(h_cand)
        
        # 候補状態の重みに関する勾配
        dWh = concat_r.T @ dh_cand_raw / batch_size
        dbh = dh_cand_raw.mean(axis=0)
        dconcat_r = dh_cand_raw @ self.Wh.T
        
        # concat_r = [r * h_prev, x_t]
        dr_h_prev = dconcat_r[:, :self.hidden_dim]
        dx_from_cand = dconcat_r[:, self.hidden_dim:]
        dr = dr_h_prev * h_prev
        dh_prev_from_r = dr_h_prev * r
        
        # 更新ゲートの勾配: z = sigmoid(...)
        dz_raw = dz * sigmoid_derivative(z)
        dWz = concat.T @ dz_raw / batch_size
        dbz = dz_raw.mean(axis=0)
        dconcat_z = dz_raw @ self.Wz.T
        
        # リセットゲートの勾配: r = sigmoid(...)
        dr_raw = dr * sigmoid_derivative(r)
        dWr = concat.T @ dr_raw / batch_size
        dbr = dr_raw.mean(axis=0)
        dconcat_r_gate = dr_raw @ self.Wr.T
        
        # concat = [h_prev, x_t] からの勾配を集約
        dconcat = dconcat_z + dconcat_r_gate
        dh_prev_from_gates = dconcat[:, :self.hidden_dim]
        dx_from_gates = dconcat[:, self.hidden_dim:]
        
        # 最終勾配
        dh_prev = dh_prev_direct + dh_prev_from_r + dh_prev_from_gates
        dx_t = dx_from_cand + dx_from_gates
        
        grads = {
            'dWz': dWz, 'dbz': dbz,
            'dWr': dWr, 'dbr': dbr,
            'dWh': dWh, 'dbh': dbh
        }
        
        return dx_t, dh_prev, grads
    
    def count_params(self):
        """パラメータ数をカウント"""
        return (self.Wz.size + self.bz.size +
                self.Wr.size + self.br.size +
                self.Wh.size + self.bh.size)


# テスト
np.random.seed(42)
cell = GRUCell(input_dim=3, hidden_dim=5)
x = np.random.randn(2, 3)  # batch=2, input=3
h = np.zeros((2, 5))       # batch=2, hidden=5

h_next = cell.forward(x, h)
print("GRUCell テスト:")
print(f"  入力:  x.shape = {x.shape}, h.shape = {h.shape}")
print(f"  出力:  h_next.shape = {h_next.shape}")
print(f"  h_next[0,:3] = {h_next[0,:3]}")
print(f"  パラメータ数: {cell.count_params()}")

In [ ]:
# ============================================================
# GRUモデル: シーケンス処理 + 学習ループ
# ============================================================

class GRUModel:
    """GRUベースの時系列予測モデル"""
    
    def __init__(self, input_dim, hidden_dim, output_dim, lr=0.01):
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.cell = GRUCell(input_dim, hidden_dim)
        
        # 出力層の重み
        scale = np.sqrt(2.0 / hidden_dim)
        self.Wy = np.random.randn(hidden_dim, output_dim) * scale
        self.by = np.zeros(output_dim)
        self.name = 'GRU'
    
    def forward_sequence(self, X):
        """
        シーケンス全体をフォワードパス
        X: (batch_size, seq_length, input_dim)
        returns: output (batch_size, output_dim), hidden states list
        """
        batch_size, seq_length, _ = X.shape
        h = np.zeros((batch_size, self.hidden_dim))
        
        self.h_list = [h]
        self.cell_caches = []
        
        for t in range(seq_length):
            h = self.cell.forward(X[:, t, :], h)
            self.cell_caches.append(self.cell.cache)
            self.h_list.append(h)
        
        # 最後の隠れ状態から出力を計算
        y = h @ self.Wy + self.by
        self.X = X
        return y
    
    def backward_sequence(self, dy):
        """シーケンス全体のバックワードパス (BPTT)"""
        batch_size = dy.shape[0]
        seq_length = self.X.shape[1]
        
        # 出力層の勾配
        dWy = self.h_list[-1].T @ dy / batch_size
        dby = dy.mean(axis=0)
        dh = dy @ self.Wy.T
        
        # GRUの勾配を蓄積
        total_grads = None
        
        for t in reversed(range(seq_length)):
            self.cell.cache = self.cell_caches[t]
            _, dh, grads = self.cell.backward(dh)
            
            # 勾配クリッピング
            dh = np.clip(dh, -5, 5)
            
            if total_grads is None:
                total_grads = {k: v.copy() for k, v in grads.items()}
            else:
                for k in total_grads:
                    total_grads[k] += grads[k]
        
        # パラメータ更新 (SGD)
        for k, v in total_grads.items():
            v_clipped = np.clip(v, -1, 1)
            if k == 'dWz': self.cell.Wz -= self.lr * v_clipped
            elif k == 'dbz': self.cell.bz -= self.lr * v_clipped
            elif k == 'dWr': self.cell.Wr -= self.lr * v_clipped
            elif k == 'dbr': self.cell.br -= self.lr * v_clipped
            elif k == 'dWh': self.cell.Wh -= self.lr * v_clipped
            elif k == 'dbh': self.cell.bh -= self.lr * v_clipped
        
        self.Wy -= self.lr * np.clip(dWy, -1, 1)
        self.by -= self.lr * np.clip(dby, -1, 1)
    
    def train_step(self, X, y_true):
        """1ステップの学習"""
        y_pred = self.forward_sequence(X)
        loss = np.mean((y_pred - y_true) ** 2)
        dy = 2 * (y_pred - y_true) / y_true.shape[0]
        self.backward_sequence(dy)
        return loss
    
    def predict(self, X):
        """予測のみ（勾配計算なし）"""
        return self.forward_sequence(X)
    
    def count_params(self):
        return self.cell.count_params() + self.Wy.size + self.by.size


# テスト
np.random.seed(42)
model = GRUModel(input_dim=1, hidden_dim=8, output_dim=1, lr=0.01)
X_test = np.random.randn(4, 10, 1)  # batch=4, seq=10, input=1
y_test = np.random.randn(4, 1)

pred = model.predict(X_test)
print("GRUModel テスト:")
print(f"  入力: X.shape = {X_test.shape}")
print(f"  出力: pred.shape = {pred.shape}")
print(f"  パラメータ数: {model.count_params()}")

# 学習テスト
loss = model.train_step(X_test, y_test)
print(f"  初期損失: {loss:.4f}")

In [ ]:
# ============================================================
# GRUの勾配チェック
# ============================================================

def numerical_gradient(model, X, y_true, param_name, epsilon=1e-5):
    """
    数値微分による勾配計算（勾配チェック用）
    """
    # パラメータへの参照を取得
    if param_name == 'Wz': param = model.cell.Wz
    elif param_name == 'Wr': param = model.cell.Wr
    elif param_name == 'Wh': param = model.cell.Wh
    elif param_name == 'Wy': param = model.Wy
    else: raise ValueError(f"Unknown param: {param_name}")
    
    grad = np.zeros_like(param)
    # ランダムに数個の要素だけチェック（高速化）
    indices = [tuple(np.random.randint(0, s) for s in param.shape) for _ in range(5)]
    
    for idx in indices:
        old_val = param[idx]
        
        param[idx] = old_val + epsilon
        y1 = model.predict(X)
        loss_plus = np.mean((y1 - y_true) ** 2)
        
        param[idx] = old_val - epsilon
        y2 = model.predict(X)
        loss_minus = np.mean((y2 - y_true) ** 2)
        
        param[idx] = old_val
        grad[idx] = (loss_plus - loss_minus) / (2 * epsilon)
    
    return grad, indices


# 勾配チェック実行
np.random.seed(42)
model_gc = GRUModel(input_dim=1, hidden_dim=4, output_dim=1, lr=0.001)
X_gc = np.random.randn(2, 5, 1)  # 小さなデータ
y_gc = np.random.randn(2, 1)

print("GRU 勾配チェック:")
print("=" * 50)

for param_name in ['Wz', 'Wr', 'Wh', 'Wy']:
    # 解析的勾配を計算
    y_pred = model_gc.forward_sequence(X_gc)
    dy = 2 * (y_pred - y_gc) / y_gc.shape[0]
    
    # バックワード前にキャッシュをコピー
    caches_backup = [c for c in model_gc.cell_caches]
    
    # 数値勾配
    num_grad, indices = numerical_gradient(model_gc, X_gc, y_gc, param_name)
    
    # 解析的勾配を再計算（数値勾配計算でキャッシュが変わるため）
    y_pred = model_gc.forward_sequence(X_gc)
    dy = 2 * (y_pred - y_gc) / y_gc.shape[0]
    
    batch_size = X_gc.shape[0]
    seq_length = X_gc.shape[1]
    dh = dy @ model_gc.Wy.T
    
    if param_name == 'Wy':
        ana_grad = model_gc.h_list[-1].T @ dy / batch_size
    else:
        total_grads = None
        for t in reversed(range(seq_length)):
            model_gc.cell.cache = model_gc.cell_caches[t]
            _, dh_temp, grads = model_gc.cell.backward(dh)
            dh = np.clip(dh_temp, -5, 5)
            if total_grads is None:
                total_grads = {k: v.copy() for k, v in grads.items()}
            else:
                for k in total_grads:
                    total_grads[k] += grads[k]
        ana_grad = total_grads[f'd{param_name}']
    
    # 比較
    errors = []
    for idx in indices:
        if num_grad[idx] != 0:
            a = ana_grad[idx]
            n = num_grad[idx]
            rel_err = abs(a - n) / (abs(a) + abs(n) + 1e-8)
            errors.append(rel_err)
    
    avg_err = np.mean(errors) if errors else 0
    status = "OK" if avg_err < 0.1 else "WARN"
    print(f"  {param_name:3s}: avg relative error = {avg_err:.6f}  [{status}]")

print("\n勾配チェック完了。relative error < 0.01 であれば実装は正しいと判断できます。")

---
## 3. RNN/LSTM/GRU 三者比較

ここでは、バニラRNN、LSTM、GRUの3つのモデルを同じインターフェースで実装し、
パラメータ数、学習速度、精度を公平に比較します。

| 特性 | バニラRNN | LSTM | GRU |
|------|-----------|------|-----|
| ゲート数 | 0 | 3+セル候補 | 2+候補 |
| 状態 | h のみ | h + c | h のみ |
| パラメータ数 | 少ない | 多い | 中間 |
| 長期依存性 | 苦手 | 得意 | 得意 |
| 学習速度 | 速い | 遅い | 中間 |

In [ ]:
# ============================================================
# RNNモデルの定義（バニラRNN）
# ============================================================

class RNNCell:
    """バニラRNNセル"""
    def __init__(self, input_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim
        scale = np.sqrt(2.0 / concat_dim)
        self.Wh = np.random.randn(concat_dim, hidden_dim) * scale
        self.bh = np.zeros(hidden_dim)
        self.cache = None
    
    def forward(self, x_t, h_prev):
        concat = np.concatenate([h_prev, x_t], axis=1)
        h_next = np.tanh(concat @ self.Wh + self.bh)
        self.cache = (x_t, h_prev, h_next, concat)
        return h_next
    
    def backward(self, dh_next):
        x_t, h_prev, h_next, concat = self.cache
        batch_size = x_t.shape[0]
        dh_raw = dh_next * tanh_derivative(h_next)
        dWh = concat.T @ dh_raw / batch_size
        dbh = dh_raw.mean(axis=0)
        dconcat = dh_raw @ self.Wh.T
        dh_prev = dconcat[:, :self.hidden_dim]
        dx_t = dconcat[:, self.hidden_dim:]
        return dx_t, dh_prev, {'dWh': dWh, 'dbh': dbh}
    
    def count_params(self):
        return self.Wh.size + self.bh.size


class RNNModel:
    """バニラRNNモデル"""
    def __init__(self, input_dim, hidden_dim, output_dim, lr=0.01):
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.cell = RNNCell(input_dim, hidden_dim)
        scale = np.sqrt(2.0 / hidden_dim)
        self.Wy = np.random.randn(hidden_dim, output_dim) * scale
        self.by = np.zeros(output_dim)
        self.name = 'RNN'
    
    def forward_sequence(self, X):
        batch_size, seq_length, _ = X.shape
        h = np.zeros((batch_size, self.hidden_dim))
        self.h_list = [h]
        self.cell_caches = []
        for t in range(seq_length):
            h = self.cell.forward(X[:, t, :], h)
            self.cell_caches.append(self.cell.cache)
            self.h_list.append(h)
        y = h @ self.Wy + self.by
        self.X = X
        return y
    
    def backward_sequence(self, dy):
        batch_size = dy.shape[0]
        seq_length = self.X.shape[1]
        dWy = self.h_list[-1].T @ dy / batch_size
        dby = dy.mean(axis=0)
        dh = dy @ self.Wy.T
        total_grads = None
        for t in reversed(range(seq_length)):
            self.cell.cache = self.cell_caches[t]
            _, dh, grads = self.cell.backward(dh)
            dh = np.clip(dh, -5, 5)
            if total_grads is None:
                total_grads = {k: v.copy() for k, v in grads.items()}
            else:
                for k in total_grads:
                    total_grads[k] += grads[k]
        for k, v in total_grads.items():
            v_clipped = np.clip(v, -1, 1)
            if k == 'dWh': self.cell.Wh -= self.lr * v_clipped
            elif k == 'dbh': self.cell.bh -= self.lr * v_clipped
        self.Wy -= self.lr * np.clip(dWy, -1, 1)
        self.by -= self.lr * np.clip(dby, -1, 1)
    
    def train_step(self, X, y_true):
        y_pred = self.forward_sequence(X)
        loss = np.mean((y_pred - y_true) ** 2)
        dy = 2 * (y_pred - y_true) / y_true.shape[0]
        self.backward_sequence(dy)
        return loss
    
    def predict(self, X):
        return self.forward_sequence(X)
    
    def count_params(self):
        return self.cell.count_params() + self.Wy.size + self.by.size


# ============================================================
# LSTMモデルの定義
# ============================================================

class LSTMCell:
    """LSTMセル"""
    def __init__(self, input_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim
        scale = np.sqrt(2.0 / concat_dim)
        # 忘却ゲート
        self.Wf = np.random.randn(concat_dim, hidden_dim) * scale
        self.bf = np.ones(hidden_dim)  # 忘却ゲートは1で初期化
        # 入力ゲート
        self.Wi = np.random.randn(concat_dim, hidden_dim) * scale
        self.bi = np.zeros(hidden_dim)
        # セル候補
        self.Wc = np.random.randn(concat_dim, hidden_dim) * scale
        self.bc = np.zeros(hidden_dim)
        # 出力ゲート
        self.Wo = np.random.randn(concat_dim, hidden_dim) * scale
        self.bo = np.zeros(hidden_dim)
        self.cache = None
    
    def forward(self, x_t, h_prev, c_prev):
        concat = np.concatenate([h_prev, x_t], axis=1)
        f = sigmoid(concat @ self.Wf + self.bf)
        i = sigmoid(concat @ self.Wi + self.bi)
        c_cand = np.tanh(concat @ self.Wc + self.bc)
        o = sigmoid(concat @ self.Wo + self.bo)
        c_next = f * c_prev + i * c_cand
        h_next = o * np.tanh(c_next)
        self.cache = (x_t, h_prev, c_prev, f, i, c_cand, o, c_next, h_next, concat)
        return h_next, c_next
    
    def backward(self, dh_next, dc_next):
        x_t, h_prev, c_prev, f, i, c_cand, o, c_next, h_next, concat = self.cache
        batch_size = x_t.shape[0]
        
        tanh_c = np.tanh(c_next)
        do = dh_next * tanh_c
        dc_next = dc_next + dh_next * o * tanh_derivative(tanh_c)
        
        df = dc_next * c_prev
        di = dc_next * c_cand
        dc_cand = dc_next * i
        dc_prev = dc_next * f
        
        df_raw = df * sigmoid_derivative(f)
        di_raw = di * sigmoid_derivative(i)
        dc_cand_raw = dc_cand * tanh_derivative(c_cand)
        do_raw = do * sigmoid_derivative(o)
        
        dWf = concat.T @ df_raw / batch_size
        dbf = df_raw.mean(axis=0)
        dWi = concat.T @ di_raw / batch_size
        dbi = di_raw.mean(axis=0)
        dWc = concat.T @ dc_cand_raw / batch_size
        dbc = dc_cand_raw.mean(axis=0)
        dWo = concat.T @ do_raw / batch_size
        dbo = do_raw.mean(axis=0)
        
        dconcat = (df_raw @ self.Wf.T + di_raw @ self.Wi.T +
                   dc_cand_raw @ self.Wc.T + do_raw @ self.Wo.T)
        dh_prev = dconcat[:, :self.hidden_dim]
        dx_t = dconcat[:, self.hidden_dim:]
        
        grads = {
            'dWf': dWf, 'dbf': dbf, 'dWi': dWi, 'dbi': dbi,
            'dWc': dWc, 'dbc': dbc, 'dWo': dWo, 'dbo': dbo
        }
        return dx_t, dh_prev, dc_prev, grads
    
    def count_params(self):
        return (self.Wf.size + self.bf.size + self.Wi.size + self.bi.size +
                self.Wc.size + self.bc.size + self.Wo.size + self.bo.size)


class LSTMModel:
    """LSTMモデル"""
    def __init__(self, input_dim, hidden_dim, output_dim, lr=0.01):
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.cell = LSTMCell(input_dim, hidden_dim)
        scale = np.sqrt(2.0 / hidden_dim)
        self.Wy = np.random.randn(hidden_dim, output_dim) * scale
        self.by = np.zeros(output_dim)
        self.name = 'LSTM'
    
    def forward_sequence(self, X):
        batch_size, seq_length, _ = X.shape
        h = np.zeros((batch_size, self.hidden_dim))
        c = np.zeros((batch_size, self.hidden_dim))
        self.h_list = [h]
        self.c_list = [c]
        self.cell_caches = []
        for t in range(seq_length):
            h, c = self.cell.forward(X[:, t, :], h, c)
            self.cell_caches.append(self.cell.cache)
            self.h_list.append(h)
            self.c_list.append(c)
        y = h @ self.Wy + self.by
        self.X = X
        return y
    
    def backward_sequence(self, dy):
        batch_size = dy.shape[0]
        seq_length = self.X.shape[1]
        dWy = self.h_list[-1].T @ dy / batch_size
        dby = dy.mean(axis=0)
        dh = dy @ self.Wy.T
        dc = np.zeros_like(dh)
        total_grads = None
        for t in reversed(range(seq_length)):
            self.cell.cache = self.cell_caches[t]
            _, dh, dc, grads = self.cell.backward(dh, dc)
            dh = np.clip(dh, -5, 5)
            dc = np.clip(dc, -5, 5)
            if total_grads is None:
                total_grads = {k: v.copy() for k, v in grads.items()}
            else:
                for k in total_grads:
                    total_grads[k] += grads[k]
        for k, v in total_grads.items():
            v_clipped = np.clip(v, -1, 1)
            if k == 'dWf': self.cell.Wf -= self.lr * v_clipped
            elif k == 'dbf': self.cell.bf -= self.lr * v_clipped
            elif k == 'dWi': self.cell.Wi -= self.lr * v_clipped
            elif k == 'dbi': self.cell.bi -= self.lr * v_clipped
            elif k == 'dWc': self.cell.Wc -= self.lr * v_clipped
            elif k == 'dbc': self.cell.bc -= self.lr * v_clipped
            elif k == 'dWo': self.cell.Wo -= self.lr * v_clipped
            elif k == 'dbo': self.cell.bo -= self.lr * v_clipped
        self.Wy -= self.lr * np.clip(dWy, -1, 1)
        self.by -= self.lr * np.clip(dby, -1, 1)
    
    def train_step(self, X, y_true):
        y_pred = self.forward_sequence(X)
        loss = np.mean((y_pred - y_true) ** 2)
        dy = 2 * (y_pred - y_true) / y_true.shape[0]
        self.backward_sequence(dy)
        return loss
    
    def predict(self, X):
        return self.forward_sequence(X)
    
    def count_params(self):
        return self.cell.count_params() + self.Wy.size + self.by.size


print("RNNModel, LSTMModel, GRUModel の定義完了")
print("\n各モデルのパラメータ数比較 (input=1, hidden=16, output=1):")
np.random.seed(42)
rnn_m = RNNModel(1, 16, 1)
lstm_m = LSTMModel(1, 16, 1)
gru_m = GRUModel(1, 16, 1)
print(f"  RNN:  {rnn_m.count_params():5d} params")
print(f"  LSTM: {lstm_m.count_params():5d} params")
print(f"  GRU:  {gru_m.count_params():5d} params")
print(f"\n  GRU/LSTM比: {gru_m.count_params()/lstm_m.count_params():.1%}")

In [ ]:
# ============================================================
# パラメータ数比較の可視化
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 異なる隠れ層サイズでのパラメータ数比較
hidden_sizes = [4, 8, 16, 32, 64]
rnn_params = []
lstm_params = []
gru_params = []

for h in hidden_sizes:
    np.random.seed(42)
    rnn_params.append(RNNModel(1, h, 1).count_params())
    lstm_params.append(LSTMModel(1, h, 1).count_params())
    gru_params.append(GRUModel(1, h, 1).count_params())

# 棒グラフ
ax = axes[0]
x = np.arange(len(hidden_sizes))
width = 0.25
ax.bar(x - width, rnn_params, width, label='RNN', color='#FF6B6B', alpha=0.8)
ax.bar(x, lstm_params, width, label='LSTM', color='#4ECDC4', alpha=0.8)
ax.bar(x + width, gru_params, width, label='GRU', color='#45B7D1', alpha=0.8)
ax.set_xlabel('Hidden Dimension', fontsize=12)
ax.set_ylabel('Parameter Count', fontsize=12)
ax.set_title('Parameter Count Comparison', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(hidden_sizes)
ax.legend(fontsize=11)
ax.set_yscale('log')

# パラメータ比率
ax = axes[1]
ratio_gru_lstm = [g/l for g, l in zip(gru_params, lstm_params)]
ratio_rnn_lstm = [r/l for r, l in zip(rnn_params, lstm_params)]
ax.plot(hidden_sizes, ratio_rnn_lstm, 'o-', color='#FF6B6B', linewidth=2, markersize=8, label='RNN/LSTM')
ax.plot(hidden_sizes, ratio_gru_lstm, 's-', color='#45B7D1', linewidth=2, markersize=8, label='GRU/LSTM')
ax.axhline(y=0.75, color='gray', linestyle='--', alpha=0.5, label='75% line')
ax.set_xlabel('Hidden Dimension', fontsize=12)
ax.set_ylabel('Ratio to LSTM', fontsize=12)
ax.set_title('Parameter Ratio vs LSTM', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print("\n【観察】")
print("  - GRUはLSTMの約75%のパラメータ数")
print("  - バニラRNNはLSTMの約25%のパラメータ数")
print("  - 隠れ層が大きくなるほど、この比率は安定する")

---
## 4. 時系列データの前処理

時系列データを機械学習モデルに入力する前に、適切な前処理が必要です。

### 重要なポイント

1. **正規化（Normalization）**: 学習の安定化のため、訓練データの統計量で正規化
   - 注意: テストデータも**訓練データの統計量**で正規化（データリーク防止）

2. **訓練/テスト分割**: 時系列データでは**シャッフルしない！**
   - 時間的順序を保持して分割する
   - 未来のデータで過去を予測するのは不正

3. **ウィンドウ作成（Sliding Window）**: 固定長の入力シーケンスとターゲットを作成
   - 入力: $[x_t, x_{t+1}, ..., x_{t+L-1}]$
   - ターゲット: $x_{t+L}$

In [ ]:
# ============================================================
# 合成時系列データの生成と前処理
# ============================================================

def generate_time_series(n_points=1000):
    """
    合成時系列データを生成
    トレンド + 季節性（2つの周期）+ ノイズ
    """
    t = np.arange(n_points)
    trend = 0.01 * t
    seasonal = np.sin(2 * np.pi * t / 50) + 0.5 * np.sin(2 * np.pi * t / 20)
    noise = np.random.randn(n_points) * 0.2
    return trend + seasonal + noise


def normalize(data, train_size):
    """
    訓練データの統計量で正規化
    注意: テストデータも訓練データの平均・標準偏差を使う
    """
    train = data[:train_size]
    mu, sigma = train.mean(), train.std()
    return (data - mu) / sigma, mu, sigma


def create_sequences(data, seq_length):
    """
    スライディングウィンドウでシーケンスを作成
    入力: [x_t, ..., x_{t+L-1}], ターゲット: x_{t+L}
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)


# データ生成
np.random.seed(42)
raw_data = generate_time_series(n_points=1000)

# パラメータ
TRAIN_SIZE = 700
SEQ_LENGTH = 20

# 正規化
normalized_data, mu, sigma = normalize(raw_data, TRAIN_SIZE)

# シーケンス作成
X_all, y_all = create_sequences(normalized_data, SEQ_LENGTH)
X_all = X_all.reshape(-1, SEQ_LENGTH, 1)  # (samples, seq_length, 1)
y_all = y_all.reshape(-1, 1)              # (samples, 1)

# 訓練/テスト分割（時系列なのでシャッフルしない！）
train_end = TRAIN_SIZE - SEQ_LENGTH
X_train, y_train = X_all[:train_end], y_all[:train_end]
X_test, y_test = X_all[train_end:], y_all[train_end:]

print("データ生成・前処理完了:")
print(f"  生データ:     {raw_data.shape}")
print(f"  正規化後:     mean={normalized_data[:TRAIN_SIZE].mean():.4f}, std={normalized_data[:TRAIN_SIZE].std():.4f}")
print(f"  訓練データ:   X={X_train.shape}, y={y_train.shape}")
print(f"  テストデータ: X={X_test.shape}, y={y_test.shape}")
print(f"  シーケンス長: {SEQ_LENGTH}")

In [ ]:
# ============================================================
# 合成時系列データの可視化
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 元データ
ax = axes[0]
ax.plot(raw_data, linewidth=0.8, color='#2C3E50')
ax.axvline(x=TRAIN_SIZE, color='red', linestyle='--', linewidth=2, label='Train/Test Split')
ax.fill_between(range(TRAIN_SIZE), raw_data[:TRAIN_SIZE].min(), raw_data[:TRAIN_SIZE].max(),
                alpha=0.1, color='blue', label='Train')
ax.fill_between(range(TRAIN_SIZE, len(raw_data)),
                raw_data[TRAIN_SIZE:].min(), raw_data[TRAIN_SIZE:].max(),
                alpha=0.1, color='orange', label='Test')
ax.set_title('Raw Time Series Data', fontsize=14)
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.legend(fontsize=10)

# 正規化後
ax = axes[1]
ax.plot(normalized_data, linewidth=0.8, color='#27AE60')
ax.axvline(x=TRAIN_SIZE, color='red', linestyle='--', linewidth=2)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.set_title('Normalized Time Series (using train statistics)', fontsize=14)
ax.set_xlabel('Time Step')
ax.set_ylabel('Normalized Value')

# 成分分解
ax = axes[2]
t = np.arange(1000)
ax.plot(0.01 * t, label='Trend', linewidth=2, alpha=0.8)
ax.plot(np.sin(2*np.pi*t/50), label='Seasonal (T=50)', linewidth=1.5, alpha=0.7)
ax.plot(0.5*np.sin(2*np.pi*t/20), label='Seasonal (T=20)', linewidth=1.5, alpha=0.7)
ax.set_title('Time Series Components', fontsize=14)
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("\n【前処理のポイント】")
print("  1. 訓練データのみで平均・標準偏差を計算 → テストデータにも同じ値を適用")
print("  2. 時系列なので時間順序を保ったまま分割（シャッフル禁止）")
print("  3. スライディングウィンドウで固定長の入力シーケンスを作成")

---
## 5. 合成時系列予測

RNN、LSTM、GRUの3モデルを同じデータで学習し、性能を比較します。

In [ ]:
# ============================================================
# 3モデルの学習
# ============================================================

def train_model(model, X_train, y_train, X_test, y_test, 
                n_epochs=100, batch_size=32, verbose=True):
    """
    ミニバッチSGDによるモデル学習
    """
    train_losses = []
    test_losses = []
    n_samples = X_train.shape[0]
    n_batches = max(1, n_samples // batch_size)
    
    start_time = time.time()
    
    for epoch in range(n_epochs):
        epoch_loss = 0
        
        # ミニバッチ学習（時系列なので順序を保つ）
        for b in range(n_batches):
            start_idx = b * batch_size
            end_idx = min(start_idx + batch_size, n_samples)
            X_batch = X_train[start_idx:end_idx]
            y_batch = y_train[start_idx:end_idx]
            
            loss = model.train_step(X_batch, y_batch)
            epoch_loss += loss
        
        train_loss = epoch_loss / n_batches
        
        # テスト損失
        y_test_pred = model.predict(X_test)
        test_loss = np.mean((y_test_pred - y_test) ** 2)
        
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        
        if verbose and (epoch + 1) % 20 == 0:
            print(f"    Epoch {epoch+1:3d}/{n_epochs}: "
                  f"Train Loss = {train_loss:.4f}, Test Loss = {test_loss:.4f}")
    
    elapsed = time.time() - start_time
    return train_losses, test_losses, elapsed


# モデル設定
HIDDEN_DIM = 16
LR = 0.01
N_EPOCHS = 100
BATCH_SIZE = 32

# 各モデルの学習
results = {}

for ModelClass, name in [(RNNModel, 'RNN'), (LSTMModel, 'LSTM'), (GRUModel, 'GRU')]:
    print(f"\n{'='*50}")
    print(f"{name} の学習開始 (hidden={HIDDEN_DIM}, lr={LR})")
    print(f"{'='*50}")
    
    np.random.seed(42)
    model = ModelClass(input_dim=1, hidden_dim=HIDDEN_DIM, output_dim=1, lr=LR)
    train_losses, test_losses, elapsed = train_model(
        model, X_train, y_train, X_test, y_test,
        n_epochs=N_EPOCHS, batch_size=BATCH_SIZE
    )
    
    results[name] = {
        'model': model,
        'train_losses': train_losses,
        'test_losses': test_losses,
        'elapsed': elapsed,
        'params': model.count_params()
    }
    
    print(f"  完了! ({elapsed:.1f}秒, {model.count_params()} params, "
          f"final test loss = {test_losses[-1]:.4f})")

print("\n" + "="*50)
print("全モデルの学習完了!")

In [ ]:
# ============================================================
# 学習曲線の比較可視化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'RNN': '#FF6B6B', 'LSTM': '#4ECDC4', 'GRU': '#45B7D1'}

# 訓練損失
ax = axes[0]
for name, res in results.items():
    ax.plot(res['train_losses'], label=name, color=colors[name], linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Train Loss (MSE)', fontsize=12)
ax.set_title('Training Loss Curves', fontsize=14)
ax.legend(fontsize=12)
ax.set_yscale('log')

# テスト損失
ax = axes[1]
for name, res in results.items():
    ax.plot(res['test_losses'], label=name, color=colors[name], linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Loss (MSE)', fontsize=12)
ax.set_title('Test Loss Curves', fontsize=14)
ax.legend(fontsize=12)
ax.set_yscale('log')

# サマリー棒グラフ
ax = axes[2]
names = list(results.keys())
final_test = [results[n]['test_losses'][-1] for n in names]
params = [results[n]['params'] for n in names]
times = [results[n]['elapsed'] for n in names]

x = np.arange(len(names))
width = 0.3

# 正規化して比較
max_loss = max(final_test)
max_params = max(params)
max_time = max(times)

bars1 = ax.bar(x - width, [l/max_loss for l in final_test], width,
               label='Test Loss (norm)', color=[colors[n] for n in names], alpha=0.6)
bars2 = ax.bar(x, [p/max_params for p in params], width,
               label='Params (norm)', color=[colors[n] for n in names], alpha=0.8)
bars3 = ax.bar(x + width, [t/max_time for t in times], width,
               label='Time (norm)', color=[colors[n] for n in names], alpha=1.0)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Normalized Value', fontsize=12)
ax.set_title('Model Comparison Summary', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

# 数値サマリー
print("\n" + "="*60)
print(f"{'Model':<8} {'Params':>8} {'Time(s)':>10} {'Train Loss':>12} {'Test Loss':>12}")
print("-"*60)
for name in names:
    r = results[name]
    print(f"{name:<8} {r['params']:>8d} {r['elapsed']:>10.2f} "
          f"{r['train_losses'][-1]:>12.6f} {r['test_losses'][-1]:>12.6f}")
print("="*60)

In [ ]:
# ============================================================
# 予測結果の可視化
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# テストデータでの予測
test_range = np.arange(train_end, train_end + len(y_test))

for idx, (name, res) in enumerate(results.items()):
    ax = axes[idx]
    model = res['model']
    y_pred = model.predict(X_test)
    
    # 逆正規化
    y_true_denorm = y_test.flatten() * sigma + mu
    y_pred_denorm = y_pred.flatten() * sigma + mu
    
    # 訓練データ（最後の100点）
    train_plot_range = np.arange(max(0, train_end-100), train_end)
    train_plot_data = raw_data[max(0, train_end-100)+SEQ_LENGTH:train_end+SEQ_LENGTH]
    if len(train_plot_range) > len(train_plot_data):
        train_plot_range = train_plot_range[:len(train_plot_data)]
    
    ax.plot(train_plot_range, train_plot_data[:len(train_plot_range)],
            color='gray', alpha=0.5, linewidth=1, label='Train Data')
    ax.plot(test_range, y_true_denorm, color='black', linewidth=1.5, label='Actual')
    ax.plot(test_range, y_pred_denorm, color=colors[name], linewidth=1.5,
            alpha=0.8, label=f'{name} Prediction')
    
    mse = np.mean((y_true_denorm - y_pred_denorm)**2)
    ax.axvline(x=train_end, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'{name} Prediction (MSE = {mse:.4f})', fontsize=14)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Value')
    ax.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()

print("\n【観察】")
print("  - 各モデルが時系列の季節性パターンをどの程度捉えているか確認")
print("  - LSTM/GRUはRNNより長期的なパターンを捉えやすい傾向がある")
print("  - GRUはLSTMより少ないパラメータで同等の性能を発揮できることが多い")

---
## 6. 多ステップ予測

### 自己回帰型多ステップ予測（Autoregressive Multi-step Prediction）

1ステップ先の予測を行い、その予測結果を入力として次のステップを予測する方法です。

```
入力: [x_1, x_2, ..., x_T] → 予測: x̂_{T+1}
入力: [x_2, ..., x_T, x̂_{T+1}] → 予測: x̂_{T+2}
入力: [x_3, ..., x̂_{T+1}, x̂_{T+2}] → 予測: x̂_{T+3}
...
```

### 注意点
- 予測誤差が**累積**していく
- ステップが進むほど、実際の値との乖離が大きくなる
- これは自己回帰予測の根本的な限界

In [ ]:
# ============================================================
# 多ステップ予測の実装
# ============================================================

def multi_step_predict(model, initial_seq, n_steps):
    """
    自己回帰型多ステップ予測
    
    Parameters:
        model: 学習済みモデル
        initial_seq: 初期シーケンス (seq_length, 1)
        n_steps: 予測ステップ数
    
    Returns:
        predictions: (n_steps,) 予測値の配列
    """
    predictions = []
    current_seq = initial_seq.copy()
    
    for _ in range(n_steps):
        # バッチ次元を追加: (1, seq_length, 1)
        X_input = current_seq.reshape(1, -1, 1)
        pred = model.predict(X_input)
        pred_val = pred[0, 0]
        predictions.append(pred_val)
        
        # シーケンスを1つシフトし、予測値を追加
        current_seq = np.roll(current_seq, -1)
        current_seq[-1] = pred_val
    
    return np.array(predictions)


# 多ステップ予測の実行
N_FUTURE_STEPS = 100

# テストデータの最初のシーケンスを使用
initial_sequence = X_test[0, :, 0]  # (seq_length,)
actual_future = y_test[:N_FUTURE_STEPS, 0]  # 実際の値

multi_predictions = {}
for name, res in results.items():
    preds = multi_step_predict(res['model'], initial_sequence, N_FUTURE_STEPS)
    multi_predictions[name] = preds
    mse = np.mean((preds[:len(actual_future)] - actual_future[:len(preds)])**2)
    print(f"{name}: Multi-step MSE (normalized) = {mse:.4f}")

print(f"\n予測ステップ数: {N_FUTURE_STEPS}")

In [ ]:
# ============================================================
# 多ステップ予測の可視化
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# --- 予測 vs 実測 ---
ax = axes[0]
steps = np.arange(N_FUTURE_STEPS)

# 初期シーケンスの表示
init_steps = np.arange(-SEQ_LENGTH, 0)
ax.plot(init_steps, initial_sequence, 'k-', linewidth=2, label='Initial Sequence')

# 実測値
ax.plot(steps, actual_future, 'k--', linewidth=2, alpha=0.7, label='Actual')

# 各モデルの予測
for name, preds in multi_predictions.items():
    ax.plot(steps, preds, color=colors[name], linewidth=1.5, alpha=0.8,
            label=f'{name} ({N_FUTURE_STEPS}-step)')

ax.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Prediction Start')
ax.set_xlabel('Time Step (from prediction start)', fontsize=12)
ax.set_ylabel('Normalized Value', fontsize=12)
ax.set_title(f'Multi-step Prediction ({N_FUTURE_STEPS} steps ahead)', fontsize=14)
ax.legend(fontsize=10, loc='upper right')

# --- 誤差の累積 ---
ax = axes[1]

for name, preds in multi_predictions.items():
    # 各ステップでの二乗誤差
    step_errors = (preds - actual_future) ** 2
    # 累積平均誤差
    cumulative_mse = np.cumsum(step_errors) / (np.arange(len(step_errors)) + 1)
    
    ax.plot(steps, step_errors, color=colors[name], linewidth=1, alpha=0.3)
    ax.plot(steps, cumulative_mse, color=colors[name], linewidth=2,
            label=f'{name} (cumulative MSE)')

ax.set_xlabel('Prediction Step', fontsize=12)
ax.set_ylabel('Squared Error', fontsize=12)
ax.set_title('Error Accumulation in Multi-step Prediction', fontsize=14)
ax.legend(fontsize=10)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print("\n【観察】")
print("  - 予測ステップが増えるにつれて誤差が累積していく")
print("  - ゲート付きモデル（LSTM/GRU）は、バニラRNNより安定した長期予測が可能")
print("  - しかし、どのモデルでも自己回帰予測では誤差が避けられない")
print("  - 実用上、多ステップ予測には別のアプローチ（direct forecast等）も検討すべき")

---
## 7. まとめ

### RNN vs LSTM vs GRU 比較表

| 特性 | バニラRNN | LSTM | GRU |
|------|-----------|------|-----|
| **ゲート数** | 0 | 3（忘却・入力・出力）+ セル候補 | 2（更新・リセット）+ 候補 |
| **状態** | $h_t$ のみ | $h_t$ と $c_t$ | $h_t$ のみ |
| **パラメータ数** | $n_h(n_h + n_x) + n_h$ | $4[n_h(n_h + n_x) + n_h]$ | $3[n_h(n_h + n_x) + n_h]$ |
| **LSTM比** | ~25% | 100% | ~75% |
| **長期依存性** | 苦手（勾配消失） | 得意 | 得意 |
| **学習速度** | 速い | 遅い | 中間 |
| **適用場面** | 短いシーケンス | 長いシーケンス、複雑なパターン | バランス重視 |

### チートシート：いつどれを使うか？

- **データが少ない** → GRU（パラメータが少なく過学習しにくい）
- **長いシーケンス・複雑な依存関係** → LSTM（セル状態による安定した勾配伝搬）
- **リアルタイム推論が必要** → GRU（計算が軽い）
- **シーケンスが短い（< 50ステップ）** → GRUまたはバニラRNNで十分
- **迷ったら** → まずGRUで試し、性能が不足ならLSTMへ

### よくある間違い

1. **時系列データをシャッフルする** → 時間的な依存関係が壊れる
2. **テストデータの統計量で正規化する** → データリーク（将来の情報を使ってしまう）
3. **多ステップ予測の誤差累積を無視する** → 実用上は数ステップ先が限界
4. **GRUとLSTMを常に同じ隠れ次元で比較する** → GRUはパラメータが少ないため、公平な比較にはパラメータ数を揃えるべき

### 学習チェックリスト

- [ ] GRUの2ゲート（更新・リセット）の役割を説明できる
- [ ] GRUの数式を書き出せる
- [ ] RNN/LSTM/GRUのパラメータ数の違いを計算できる
- [ ] 時系列データの正規化とウィンドウ作成を実装できる
- [ ] 自己回帰型多ステップ予測を実装できる
- [ ] 各モデルの使い分けの指針を説明できる

### 次のステップ
→ **Notebook 125: Seq2Seq（Sequence-to-Sequence）** で、エンコーダ・デコーダアーキテクチャを学びます。

---
## 8. 自己評価クイズ

以下のクイズで理解度を確認しましょう。

---

### Q1: GRUとLSTMの構造上の最も大きな違いは何ですか？

<details>
<summary>回答を表示</summary>

**GRUはセル状態 ($c_t$) を持たず、隠れ状態 ($h_t$) のみで記憶を管理する点が最大の違いです。**

具体的な違い：
- LSTM: 3つのゲート（忘却・入力・出力）＋セル候補、隠れ状態 $h_t$ とセル状態 $c_t$ の2つの状態
- GRU: 2つのゲート（更新・リセット）＋候補状態、隠れ状態 $h_t$ のみ

GRUの更新ゲート $z_t$ は、LSTMの忘却ゲートと入力ゲートの役割を1つに統合しています。
$(1-z_t)$ が「忘れる割合」、$z_t$ が「新しい情報を取り入れる割合」を同時に制御します。
</details>

---

### Q2: GRUはなぜ別個のセル状態を必要としないのですか？

<details>
<summary>回答を表示</summary>

**GRUの更新ゲートの仕組みにより、隠れ状態自体が長期記憶の役割も果たせるからです。**

LSTMでは：
- セル状態 $c_t$: 長期記憶を保持（忘却ゲートで制御）
- 隠れ状態 $h_t$: 出力ゲートで $c_t$ をフィルタリングした短期的な表現

GRUでは：
- 更新ゲート $z_t$ が $(1-z_t) \cdot h_{t-1} + z_t \cdot \tilde{h}_t$ という形で前の状態と新しい候補を線形補間
- $z_t \approx 0$ のとき、$h_t \approx h_{t-1}$ となり、情報が変化なしで伝搬する（=長期記憶）
- これはLSTMのセル状態における「忘却ゲート ≈ 1」の状態と等価

つまり、更新ゲートの設計により、セル状態を分離しなくても長期依存性を捉えられます。
</details>

---

### Q3: 自己回帰型多ステップ予測で誤差はどのように振る舞いますか？その理由は？

<details>
<summary>回答を表示</summary>

**誤差は予測ステップ数とともに累積的に増大します。**

理由：
1. 各ステップでわずかな予測誤差が生じる
2. その誤差を含んだ値が次のステップの入力として使われる
3. 誤差が入力に混入すると、次の予測もずれる
4. これが連鎖的に繰り返される（エラーの雪だるま効果）

対策：
- Direct multi-step forecast: 各ステップを直接予測する別々のモデルを訓練
- Teacher forcing: 学習時に実際の値を入力として使う
- 予測ステップ数を制限する
- アンサンブル予測で不確実性を推定する
</details>

---

### Q4: GRUをLSTMより優先すべきケースはどのようなときですか？

<details>
<summary>回答を表示</summary>

以下のケースでGRUが有利です：

1. **データセットが小さい場合**: GRUはLSTMの約75%のパラメータ数なので、過学習のリスクが低い
2. **推論速度が重要な場合**: ゲートが少なく計算が軽いため、リアルタイムアプリケーションに適している
3. **シーケンスが比較的短い場合（~50ステップ以下）**: 短いシーケンスではLSTMの複雑な構造が冗長
4. **ハイパーパラメータ探索の時間が限られている場合**: パラメータが少ないため、探索空間が小さい
5. **リソースが限られた環境（エッジデバイス等）**: メモリ使用量が少ない

逆に、非常に長いシーケンス（数百ステップ以上）や複雑な依存パターンがある場合は、LSTMの方が有利な場合があります。

**実践的なアドバイス**: まずGRUで試し、性能が不十分ならLSTMに切り替える、というアプローチが効率的です。
</details>

---
## 参考文献

1. **Cho, K., et al.** (2014). "Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation." *EMNLP 2014*. (GRUの原論文)
2. **Chung, J., et al.** (2014). "Empirical Evaluation of Gated Recurrent Neural Networks on Sequence Modeling." *NIPS 2014 Workshop*. (GRU vs LSTM比較)
3. **Hochreiter, S. & Schmidhuber, J.** (1997). "Long Short-Term Memory." *Neural Computation*. (LSTM原論文)
4. **Greff, K., et al.** (2017). "LSTM: A Search Space Odyssey." *IEEE Transactions on Neural Networks and Learning Systems*. (LSTMバリアント比較)
5. **Goodfellow, I., Bengio, Y., & Courville, A.** (2016). "Deep Learning." *MIT Press*. Chapter 10: Sequence Modeling.